# ConsistEdit → Modular Diffusers — smoke (training-free editing, adjustable structural consistency)

Masked text editing on FLUX with pre-attention vision-token fusion (ConsistEdit, arXiv:2510.17803). Publish PRIVATE `remyxai/consistedit-flux-modular` → load via `trust_remote_code` → assert `ConsistEditBlock` → a tiny masked edit → the no-op spike (consistency OFF == the stock denoise; all-keep mask == reconstructs the source). Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf scikit-image

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16


## 2 · Publish PRIVATE (upload block.py first)

In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"ConsistEditBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.ConsistEditBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"ConsistEditBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/consistedit-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))


## 3 · Load + a source image + mask

In [ ]:
from diffusers import ModularPipeline
from PIL import Image, ImageDraw
from io import BytesIO
import requests
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/consistedit-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "ConsistEditBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect ConsistEditBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

IMG_URL = "https://raw.githubusercontent.com/fallenshock/FlowEdit/main/inputs/cat.png"  #@param {type:"string"}
try:
    src = Image.open(BytesIO(requests.get(IMG_URL, timeout=30).content)).convert("RGB")
except Exception as e:
    print("fetch failed, upload one:", e)
    from google.colab import files; up=files.upload(); src=Image.open(list(up.keys())[0]).convert("RGB")
src = src.resize((1024,1024)); src.save("src.png")

# default edit mask: a box over the central subject (white = edit, black = keep). Upload your own for real use.
mask = Image.new("L", (1024,1024), 0)
ImageDraw.Draw(mask).rectangle([256, 256, 768, 768], fill=255)
mask.save("mask.png")
print("source | mask:"); display(src.resize((320,320))); display(mask.resize((320,320)))


## 4 · Milestone A — smoke (tiny masked edit)

In [ ]:
import torch
g = torch.Generator(DEV).manual_seed(0)
sm = pipe(image="src.png", mask="mask.png", source_prompt="a cat", prompt="a dog",
          height=512, width=512, T_steps=12, consistency_strength=1.0, generator=g).images[0]
assert sm.size == (512, 512), sm.size
print("[SMOKE] ran; output size", sm.size)
from IPython.display import display; display(sm)


## 5 · Milestone B — the seam spikes (no-op when off)

Two controls gate the capture/fusion seam before the full e2e:

1. **Seam OFF behaves like the stock denoise.** `consistency_strength=0` disarms the seam (`joint_attention_kwargs=None`, the processor is never driven), so the run is a plain RF-inversion edit — it should still *edit* (L1 vs source clearly > 0), and repeating it under the same seed is bit-exact (nothing stochastic leaked into the disarmed path).
2. **Seam ON + an all-keep mask holds the source.** With nothing inside the edit mask, the source Q/K/V replace the target's for *every* vision token at *every* step, so the denoise cannot move: the output reconstructs the source rather than applying the edit (L1 far below the seam-off run). This exercises the exact capture/fuse path the e2e relies on.

In [ ]:
import numpy as np, torch
from PIL import Image
src_a = np.asarray(src.resize((512,512))).astype(np.float32)/255.0
def l1_vs_src(im):
    a = np.asarray(im.resize((512,512))).astype(np.float32)/255.0
    return float(np.abs(a - src_a).mean())

# (1) consistency OFF -> the seam is disarmed (joint_attention_kwargs=None, processor never
#     driven) -> the run is the plain stock denoise, so it EDITS: it must differ from the source.
off = pipe(image="src.png", mask=None, source_prompt="a cat", prompt="a dog",
           height=512, width=512, T_steps=12, consistency_strength=0.0,
           generator=torch.Generator(DEV).manual_seed(0)).images[0]
d_off = l1_vs_src(off)
print(f"[NO-OP SPIKE] seam OFF  : L1 vs source = {d_off:.4f} (a plain edit -> should be clearly > 0)")
# ...and repeating it with a fresh generator of the same seed is bit-exact, i.e. nothing
# stochastic leaked into the disarmed path.
off2 = pipe(image="src.png", mask=None, source_prompt="a cat", prompt="a dog",
            height=512, width=512, T_steps=12, consistency_strength=0.0,
            generator=torch.Generator(DEV).manual_seed(0)).images[0]
d_rep = float(np.abs(np.asarray(off,dtype=np.float32)-np.asarray(off2,dtype=np.float32)).mean())
print(f"[NO-OP SPIKE] seam OFF  : repeat |dL1| = {d_rep:.6f} (0 = the disarmed run is deterministic/bit-exact)")

# (2) all-keep mask + alpha=1 -> the source Q/K/V replace the target's for EVERY vision token at
#     EVERY step, so the denoise cannot move: it reconstructs the source instead of editing.
keep = Image.new("L", (1024,1024), 0); keep.save("keep_all.png")
recon = pipe(image="src.png", mask="keep_all.png", source_prompt="a cat", prompt="a dog",
             height=512, width=512, T_steps=12, consistency_strength=1.0,
             generator=torch.Generator(DEV).manual_seed(0)).images[0]
d_on = l1_vs_src(recon)
print(f"[NO-OP SPIKE] seam ON   : all-keep L1 vs source = {d_on:.4f} (should be << the seam-off L1 above)")
print(f"[NO-OP SPIKE] verdict   : seam holds the source ({'PASS' if d_on < d_off else 'REVIEW'})")


## Verdict
`loaded block: ConsistEditBlock` + a coherent tiny edit + `|dL1| ≈ 0` (seam disarmed) + a low all-keep reconstruction L1 = the modular ConsistEdit seam works. Then run `e2e.ipynb` for the quantitative consistency check, the α sweep and the head-to-head vs FlowEdit / KV-Edit.